In [1]:
from langgraph.graph import StateGraph,START,END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [3]:
class CrashState(TypedDict):
    input   :   str
    step_1  :   str
    step_2  :   str
    step_3  :   str
    

In [4]:
def Step_1(state : CrashState):
    print("✅ Step - 1 Executed")
    return {"step_1":"Done", 'input' : state['input']}


def Step_2(state : CrashState):
    print("⏳ Step-2 Hanging ...Now manually interrupt from the notebook toolbar (STOP Button)")
    time.sleep(1000)
    return {"step_2":"Done"}


def Step_3(state : CrashState):
    print("✅ Step - 3 Executed")
    return {"Done" : True}



In [5]:
graph=StateGraph(CrashState)

graph.add_node("step_1",Step_1)
graph.add_node("step_2",Step_2)
graph.add_node("step_3",Step_3)

graph.set_entry_point("step_1")
graph.add_edge("step_1","step_2")
graph.add_edge("step_2","step_3")
graph.add_edge("step_3",END)

checkpointer=InMemorySaver()
buildedGraph=graph.compile(checkpointer=checkpointer)


In [8]:
try:
    print("▶️ Running Graph : Manually iterrupt Step 2 ...")
    buildedGraph.invoke({"input":"start"},config={"configurable":{"thread_id":"thread-1"}})
    
except KeyboardInterrupt:
    print("❌ Manual Interruption (Crash Simulated)")    

▶️ Running Graph : Manually iterrupt Step 2 ...
✅ Step - 1 Executed
⏳ Step-2 Hanging ...Now manually interrupt from the notebook toolbar (STOP Button)
❌ Manual Interruption (Crash Simulated)


In [ ]:
#   Resume
print("\n🔁 Re- Running the graph to demonstrate fault tolerance...")
final_state = buildedGraph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re- Running the graph to demonstrate fault tolerance...


AttributeError: 'StateGraph' object has no attribute 'invoke'